##### Import General Functions

In [6]:
from scipy.integrate import solve_ivp
import numpy as np
import pandas as pd

#### Import Functions and Dictionaries

In [7]:
from ODESystem_10272026 import SnyderODE

In [11]:
def ODEfunction(
    NUMYEARS,
    time_points,  
    init_cond,
    dictionary,
    function):

    #Herbivore Initial Conditions (eq values from baseline runs)
    B0, G0, S0 = init_cond['B0'], init_cond['G0'], init_cond['S0']

    #Benthos Initial Conditions-----------------------------------------------------
    C0, M0, T0 = init_cond['C0'], init_cond['M0'], init_cond['T0']


    #Benthos Arrays-----------------------------------------------------
    rM_array  = dictionary['rM_array']
    hB_array  = dictionary['hB_array']
    
    outputs = []
    

    
        
    for rM in rM_array:
        for hB in hB_array:

        #for C_init in np.linspace(0.01,0.99,2):
    
            #Benthos Initial Conditions-----------------------------------------------------
            #C0, M0, T0 = C_init, ((1-C_init)/2), ((1-C_init)/2) 
    
            #Benthos Parameters-----------------------------------------------------
            rC  = dictionary['rC']
            μC  = dictionary['μC']
            rM  = rM
            rT  = dictionary['rT']
            αCT = dictionary['αCT']
            αMT = dictionary['αMT']
            αMC = dictionary['αMC']  
                    
                    
            #Herbivore Parameters-----------------------------------------------------
            #Browser
            sB = dictionary['sB']
            hB = hB
            zB = dictionary['zB']
            μB = dictionary['μB']
                
            #Grazer
            sG = dictionary['sG']
            hG = dictionary['hG']
            zG = dictionary['zG']
            μG = dictionary['μG']
    
            #Scraper
            sS = dictionary['sS']
            hS = dictionary['hS']
            zS = dictionary['zS']
            μS = dictionary['μS']
            
                                        
            #Solve ODE-----------------------------------------------------               
            sol = solve_ivp(function, [0,NUMYEARS], [C0, M0, T0, B0, G0, S0], 
                                            t_eval = time_points,
                                            method = 'RK45', 
                                            args = (rC, rM, rT, αCT, αMT, αMC, μC,              #benthos
                                                   sB, hB, zB, μB,                              #browsers
                                                   sG, hG, zG, μG,                              #grazers
                                                   sS, hS, zS, μS),                             #scrapers   
                                            dense_output=True)
    
            N = sol.y
            
            C_array = N[0,:]
            M_array = N[1,:]
            T_array = N[2,:]
            
            B_array = N[3,:]
            G_array = N[4,:]
            S_array = N[5,:]
            
                
            #Ensure Run Reached Equilibrium-----------------------------------------------------
            thr_eq = 1e-02
    
            last5 = max(2, int(0.05 * N.shape[1]))
            dX = np.diff(N[:, -last5:], axis=1)
            equil = np.all(np.abs(dX) < thr_eq)
    
            if not equil:
                failed = []
                if not np.all(np.abs(np.diff(N[0, -last5:])) < thr_eq): failed.append("C")
                if not np.all(np.abs(np.diff(N[1, -last5:])) < thr_eq): failed.append("M")
                if not np.all(np.abs(np.diff(N[2, -last5:])) < thr_eq): failed.append("T")
                if not np.all(np.abs(np.diff(N[3, -last5:])) < thr_eq): failed.append("B")
                if not np.all(np.abs(np.diff(N[4, -last5:])) < thr_eq): failed.append("G")
                if not np.all(np.abs(np.diff(N[5, -last5:])) < thr_eq): failed.append("S")
        
                            
            #Categorize State at Equilibrium-----------------------------------------------------
            thr_st = 0.1
    
            Cp = C_array[-1]
            Mp = M_array[-1]
            Tp = T_array[-1]
            Fp = max(0.0, 1 - (Cp + Mp + Tp))

            #Benthos
            C_t = Cp >= thr_st
            M_t = Mp >= thr_st
            T_t = Tp >= thr_st
    
            if C_t and not M_t and not T_t:
                ben_col = "allC"
    
            elif M_t and not C_t and not T_t:
                ben_col = "allM"
        
            elif T_t and not C_t and not M_t:
                ben_col = "allT"
                    
            elif not C_t and not M_t and not T_t:
                    ben_col = "allF"
                
            else:
                dom_val = max(Cp, Mp, Tp, Fp)
        
                if dom_val == Cp:
                    ben_col = "mostlyC" if T_t else "mostlyC_noT"
            
                elif dom_val == Mp:
                    ben_col = "mostlyM" if T_t else "mostlyM_noT"
            
                elif dom_val == Tp:
                    ben_col = "mostlyT"
    
                elif dom_val == Fp:
                    ben_col = "mostlyF"
                    
                else:
                    ben_col = "other"
        
            #Herbivores
            Bp = B_array[-1]
            Gp = G_array[-1]
            Sp = S_array[-1]
    
            B_t = Bp >= thr_st
            G_t = Gp >= thr_st
            S_t = Sp >= thr_st
    
            if B_t and not G_t and not S_t:
                herb_col = "allB"
    
            elif G_t and not B_t and not S_t:
                herb_col = "allG"
            
            elif S_t and not B_t and not G_t:
                herb_col = "allS"
                
            else:
                dom_val = max(Bp, Gp, Sp)
                    
                if dom_val == Bp:
                    herb_col = "mostlyB"
            
                elif dom_val == Gp:
                            herb_col = "mostlyG"
                    
                elif dom_val == Sp:
                    herb_col = "mostlyS"
    
                else:
                    herb_col = "other"
    
            
            #Save Results-----------------------------------------------------
            benthos  = (rC, αCT, μC, rM, αMT, αMC, rT)
            herbivores = (sB, hB, zB, μB,
                          sG, hG, zG, μG,
                          sS, hS, zS, μS)
            state_eq = (C0, M0, T0, B0, G0, S0, 
                        Cp, Mp, Tp, Bp, Gp, Sp)
            labels   = (equil, ben_col, herb_col)
            arrays = (C_array, M_array, T_array, B_array, G_array, S_array)
        
                                    
            outputs.append(
                benthos
                + herbivores
                + state_eq
                + labels
                + arrays
             )
    
                
    benthos_c = ('rC', 'αCT', 'μC', 'rM', 'αMT', 'αMC', 'rT')
    herbivores_c = ('sB', 'hB', 'zB','μB',
                    'sG', 'hG', 'zG','μG',
                    'sS', 'hS', 'zS','μS')
    state_eq_c = ('C0', 'M0', 'T0', 'B0', 'G0', 'S0', 
                  'Cp', 'Mp', 'Tp', 'Bp', 'Gp', 'Sp')
    labels_c = ('equil', 'ben_col', 'herb_col')
    arrays_c = ('C_array', 'M_array', 'T_array', 'B_array', 'G_array', 'S_array')


    columns = (
        benthos_c
        + herbivores_c
        + state_eq_c
        + labels_c
        + arrays_c

    )

        
    df_prime = pd.DataFrame(outputs, columns=columns)
          
    return(df_prime)